In [ ]:
import os
import datetime
import pygame
import random
import math
import time
import numpy as np
import json
import threading
from collections import deque
import cv2
import serial as pyserial
from pygame._sdl2.video import Window, Renderer, Texture
import tkinter as tk
from tkinter import ttk
from stable_baselines3 import PPO
from scipy.signal import lfilter

# --- New Dependencies for Robust Gaze Tracking ---
import mediapipe as mp
from sklearn.svm import SVR

# -----------------------------------------------------------
# Gaze Input Integration (Using MediaPipe)
# -----------------------------------------------------------
USE_GAZE_INPUT = True
gaze_estimator = None
gaze_position = None
gaze_lock = threading.Lock()

# --- Helper Class: OneEuroFilter for Smoothing ---
class OneEuroFilter:
    """A simple and effective filter for noisy real-time signals."""
    def __init__(self, min_cutoff=1.0, beta=0.007, d_cutoff=1.0):
        self.min_cutoff = float(min_cutoff)
        self.beta = float(beta)
        self.d_cutoff = float(d_cutoff)
        self.x_prev = None
        self.dx_prev = None
        self.t_prev = None

    def __call__(self, x):
        t_now = time.time()
        
        if self.x_prev is None:
            self.t_prev = t_now
            self.x_prev = x
            self.dx_prev = np.zeros_like(x)
            return x

        dt = t_now - self.t_prev
        if dt <= 1e-6:
            return self.x_prev
            
        freq = 1.0 / dt
        dx = (x - self.x_prev) / dt
        
        alpha_d = self._alpha(self.d_cutoff, freq)
        dx_hat = self._ema(alpha_d, dx, self.dx_prev)
        
        cutoff = self.min_cutoff + self.beta * np.linalg.norm(dx_hat)
        alpha = self._alpha(cutoff, freq)
        x_hat = self._ema(alpha, x, self.x_prev)

        self.x_prev = x_hat
        self.dx_prev = dx_hat
        self.t_prev = t_now
        return x_hat

    def _alpha(self, cutoff, freq):
        tau = 1.0 / (2 * math.pi * cutoff)
        te = 1.0 / freq
        return 1.0 / (1.0 + tau / te)

    def _ema(self, alpha, x, x_prev):
        return alpha * x + (1.0 - alpha) * x_prev

# --- Main Gaze Estimator using MediaPipe ---
class RobustGazeEstimator:
    """
    Extracts gaze features using MediaPipe Face Mesh and trains a regression model.
    Features include normalized pupil position and head pose proxies.
    """
    def __init__(self):
        self.face_mesh = mp.solutions.face_mesh.FaceMesh(
            max_num_faces=1,
            refine_landmarks=True,  # Crucial for iris tracking
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.model_x = None
        self.model_y = None
        self.calibration_data_x = []
        self.calibration_data_y = []

    def extract_features(self, frame_bgr):
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False # Performance enhancement
        results = self.face_mesh.process(frame_rgb)
        
        if not results.multi_face_landmarks:
            return None

        landmarks = results.multi_face_landmarks[0].landmark
        
        # --- Head Pose Estimation ---
        nose_tip = np.array([landmarks[1].x, landmarks[1].y])
        left_eye_center = np.mean([[landmarks[i].x, landmarks[i].y] for i in [33, 160, 158, 133, 153, 144]], axis=0)
        right_eye_center = np.mean([[landmarks[i].x, landmarks[i].y] for i in [362, 385, 387, 263, 373, 380]], axis=0)
        
        head_yaw_proxy = nose_tip[0] - (left_eye_center[0] + right_eye_center[0]) / 2
        head_pitch_proxy = nose_tip[1] - (left_eye_center[1] + right_eye_center[1]) / 2

        # --- Eye-in-Head Feature (Pupil position relative to eye corners) ---
        # Using both eyes for more robust features
        # Left eye landmarks (user's left, image right)
        l_pupil_center_lm = landmarks[468]
        l_eye_left_corner_lm = landmarks[263]
        l_eye_right_corner_lm = landmarks[362]
        l_eye_top_lm = landmarks[386]
        l_eye_bottom_lm = landmarks[374]

        # Right eye landmarks (user's right, image left)
        r_pupil_center_lm = landmarks[473]
        r_eye_left_corner_lm = landmarks[133]
        r_eye_right_corner_lm = landmarks[33]
        r_eye_top_lm = landmarks[159]
        r_eye_bottom_lm = landmarks[145]

        def get_norm_pos(pupil, left, right, top, bottom):
            width = right.x - left.x
            height = bottom.y - top.y
            if abs(width) < 1e-6 or abs(height) < 1e-6: return 0.5, 0.5
            return (pupil.x - left.x) / width, (pupil.y - top.y) / height
        
        l_pupil_x, l_pupil_y = get_norm_pos(l_pupil_center_lm, l_eye_left_corner_lm, l_eye_right_corner_lm, l_eye_top_lm, l_eye_bottom_lm)
        r_pupil_x, r_pupil_y = get_norm_pos(r_pupil_center_lm, r_eye_left_corner_lm, r_eye_right_corner_lm, r_eye_top_lm, r_eye_bottom_lm)

        # Average features from both eyes
        pupil_x_norm = (l_pupil_x + r_pupil_x) / 2
        pupil_y_norm = (l_pupil_y + r_pupil_y) / 2
        
        features = np.array([pupil_x_norm, pupil_y_norm, head_yaw_proxy, head_pitch_proxy])
        return features

    def add_calibration_point(self, features, screen_coords):
        if features is not None:
            self.calibration_data_x.append((features, screen_coords[0]))
            self.calibration_data_y.append((features, screen_coords[1]))
            print(f"Added calibration point {len(self.calibration_data_x)} at {screen_coords}")
        else:
            print("Failed to add calibration point: No face detected.")

    def train_model(self):
        if len(self.calibration_data_x) < 5:
            print("Not enough calibration data to train.")
            return False
            
        X_x = np.array([item[0] for item in self.calibration_data_x])
        Y_x = np.array([item[1] for item in self.calibration_data_x])
        
        X_y = np.array([item[0] for item in self.calibration_data_y])
        Y_y = np.array([item[1] for item in self.calibration_data_y])
        
        self.model_x = SVR(kernel='rbf').fit(X_x, Y_x)
        self.model_y = SVR(kernel='rbf').fit(X_y, Y_y)
        
        print("✅ Calibration model trained successfully.")
        return True

    def predict(self, features):
        if features is None or self.model_x is None or self.model_y is None: return None
        features = features.reshape(1, -1)
        x = self.model_x.predict(features)[0]
        y = self.model_y.predict(features)[0]
        return np.array([x, y])

def run_9_point_calibration_mediapipe(estimator, screen, renderer, font):
    """
    Displays 9 points for calibration and trains the gaze model.
    Returns True on success, False on failure or user quit.
    """
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open camera.")
        return False

    w, h = screen.size
    margin = 0.1
    points = [
        (w * margin, h * margin), (w * 0.5, h * margin), (w * (1 - margin), h * margin),
        (w * margin, h * 0.5),   (w * 0.5, h * 0.5),   (w * (1 - margin), h * 0.5),
        (w * margin, h * (1 - margin)), (w * 0.5, h * (1 - margin)), (w * (1 - margin), h * (1 - margin))
    ]

    for i, point in enumerate(points):
        calibrating = True
        while calibrating:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    print("Calibration cancelled by user.")
                    cap.release()
                    return False
                
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_SPACE:
                        ret, frame = cap.read()
                        if ret:
                            features = estimator.extract_features(frame)
                            estimator.add_calibration_point(features, point)
                            calibrating = False
                        else:
                            print("Failed to capture frame.")
                    elif event.key == pygame.K_ESCAPE:
                        print("Calibration cancelled by user.")
                        cap.release()
                        return False
            
            renderer.draw_color = (0, 0, 0, 255)
            renderer.clear()
            
            text_surf = font.render(f"Look at the circle and press SPACE ({i+1}/{len(points)})", True, (255, 255, 255))
            text_rect = text_surf.get_rect(center=(w / 2, h / 2 - 50))
            text_texture = Texture.from_surface(renderer, text_surf)
            # THE FIX IS HERE: Use texture.draw() instead of renderer.copy()
            text_texture.draw(dstrect=text_rect)

            temp_surf = pygame.Surface((30, 30), pygame.SRCALPHA)
            pygame.draw.circle(temp_surf, (255, 0, 0), (15, 15), 15)
            pygame.draw.circle(temp_surf, (255, 255, 255), (15, 15), 5)
            point_texture = Texture.from_surface(renderer, temp_surf)
            # THE FIX IS HERE AS WELL
            point_texture.draw(dstrect=(int(point[0]-15), int(point[1]-15), 30, 30))

            renderer.present()
            pygame.time.wait(10)
    
    cap.release()
    return estimator.train_model()

def gaze_tracker_thread():
    """
    Continuously gets gaze data from the camera in a separate thread.
    """
    global gaze_position
    cap = cv2.VideoCapture(0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gaze_filter = OneEuroFilter()

    while True:
        if gaze_estimator is None:
            time.sleep(0.1)
            continue

        ret, frame = cap.read()
        if not ret: continue

        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        processed_frame = clahe.apply(gray_frame)
        processed_frame_bgr = cv2.cvtColor(processed_frame, cv2.COLOR_GRAY2BGR)
        
        features = gaze_estimator.extract_features(processed_frame_bgr)
        if features is not None:
            raw_pos = gaze_estimator.predict(features)
            if raw_pos is not None:
                filtered_pos = gaze_filter(raw_pos)
                with gaze_lock:
                    gaze_position = (filtered_pos[0], filtered_pos[1])
        time.sleep(0.01)

# -----------------------------------------------------------
# Force Sensor Integration
# -----------------------------------------------------------
USE_FORCE_SENSOR = False
force_sensor_input = [0.0, 0.0]
FORCE_SENSOR_AVAILABLE = False
FORCE_SENSOR_SCALE_X = 5.0
FORCE_SENSOR_SCALE_Y = 5.0
XY_FORCE_CAL = 1.33
Z_SMOOTH_ALPHA = 0.2
skip_remaining_seeds = False
current_gamma_mode = None
gamma_mode_index = 0
sample_prev_fs = None
sample_curr_fs = None
sample_lock_fs = threading.Lock()
last_fx_smooth = None
last_fy_smooth = None
HIGH_GAMMA_THRESHOLD = 0.65
expected_goal_idx = 0

try:
    ser = pyserial.Serial('COM5', 115200, timeout=0.01)
    FORCE_SENSOR_AVAILABLE = True
    print("Force sensor connected successfully.")
except Exception as e:
    print("Error: Could not open serial port:", e)
    ser = None
    FORCE_SENSOR_AVAILABLE = False
    print("Force sensor not available. Using keyboard/joystick controls only.")

def serial_reader_fs():
    global sample_prev_fs, sample_curr_fs, last_fx_smooth, last_fy_smooth, force_sensor_input
    while True:
        if ser is None:
            time.sleep(0.1)
            continue
        try:
            line = ser.readline().decode('utf-8').strip()
            if not line:
                continue
            tokens = line.split(',')
            if len(tokens) != 2:
                continue
            try:
                values = list(map(float, tokens))
            except Exception:
                continue
            sample = {
                'fx': -values[0],
                'fy': values[1],
                'timestamp': time.time()
            }
            with sample_lock_fs:
                if sample_curr_fs is None:
                    sample_curr_fs = sample
                    sample_prev_fs = sample
                else:
                    sample_prev_fs = sample_curr_fs
                    sample_curr_fs = sample

            with sample_lock_fs:
                sp = sample_prev_fs
                sc = sample_curr_fs
            now = time.time()
            if sp is None or sc is None:
                continue
            t0 = sp['timestamp']
            t1 = sc['timestamp']
            fraction = 1.0 if t1 == t0 else (now - t0) / (t1 - t0)
            fraction = max(0.0, min(1.0, fraction))
            fx_interp = sp['fx'] * (1 - fraction) + sc['fx'] * fraction
            fy_interp = sp['fy'] * (1 - fraction) + sc['fy'] * fraction

            if last_fx_smooth is None:
                smoothed_fx = fx_interp
            else:
                smoothed_fx = last_fx_smooth + Z_SMOOTH_ALPHA * (fx_interp - last_fx_smooth)
            if last_fy_smooth is None:
                smoothed_fy = fy_interp
            else:
                smoothed_fy = last_fy_smooth + Z_SMOOTH_ALPHA * (fy_interp - last_fy_smooth)
            last_fx_smooth = smoothed_fx
            last_fy_smooth = smoothed_fy

            final_fx = smoothed_fx * XY_FORCE_CAL
            final_fy = smoothed_fy * XY_FORCE_CAL
            final_fx *= FORCE_SENSOR_SCALE_X
            final_fy *= FORCE_SENSOR_SCALE_Y
            force_sensor_input = [final_fx, final_fy]
        except Exception as e:
            print("Serial reader error:", e)
            time.sleep(0.01)

if FORCE_SENSOR_AVAILABLE:
    threading.Thread(target=serial_reader_fs, daemon=True).start()

###############################################################################
# Multi-Seed Setup
###############################################################################
SCENARIO_SEEDS = [0, 2, 58]

###############################################################################
# Global flags & constants
###############################################################################
USE_AI_CONTROL = False

def eeg_noise():
    return np.random.normal(0, 190)

def burst_noise():
    return np.random.choice([-1, 1]) * np.random.normal(0, 190) * (np.random.rand() < 0.2)

def pink_noise():
    a = [1, -0.95]
    return lfilter([1], a, np.random.normal(0, 290, size=1))[0]

NOISE_FUNCTION = pink_noise

###############################################################################
# Config / Constants
###############################################################################
GAME_AREA_SIZE = (1200, 800)
FULL_VIEW_SIZE = (1600, 800)
RED_ONLY_SIZE = (1600, 800)
GAME_AREA_X = (FULL_VIEW_SIZE[0] - GAME_AREA_SIZE[0]) // 2
GAME_AREA_Y = 0
NOISE_MAGNITUDE = 1
MIN_NOISE, MAX_NOISE, NOISE_STEP = 0.0, 2.0, 0.1
OLD_WINDOW_SIZE = (600, 600)
SCALING_FACTOR_X = GAME_AREA_SIZE[0] / OLD_WINDOW_SIZE[0]
SCALING_FACTOR_Y = GAME_AREA_SIZE[1] / OLD_WINDOW_SIZE[1]
SCALING_FACTOR = (SCALING_FACTOR_X + SCALING_FACTOR_Y) / 2

WHITE, BLACK, RED, GREEN, BLUE, YELLOW, ORANGE, GRAY, LIGHT_GRAY, DARK_GRAY = (255, 255, 255), (0, 0, 0), (255, 60, 60), (60, 180, 60), (60, 120, 255), (240, 230, 60), (255, 140, 0), (128, 128, 128), (200, 200, 200), (80, 80, 80)
TEXT_COLOR, HIGHLIGHT_COLOR, EXPECTED_GOAL_COLOR, WRONG_GOAL_COLOR, HISTORY_COLOR = (30, 30, 40), (70, 70, 230), (0, 200, 100), (255, 0, 0), (100, 100, 115)
BACKGROUND_COLOR = (250, 240, 210)
FONT_COLOR = TEXT_COLOR
FONT_SIZE = int(18 * SCALING_FACTOR)
TITLE_FONT_SIZE = int(22 * SCALING_FACTOR)
ARROW_LENGTH = int(60 * SCALING_FACTOR)
OBSTACLE_RADIUS = int(10 * SCALING_FACTOR)
COLLISION_BUFFER = int(5 * SCALING_FACTOR)
ENABLE_OBSTACLES = True
MAX_SPEED = 3 * SCALING_FACTOR
DOT_RADIUS = int(15 * SCALING_FACTOR)
TARGET_RADIUS = int(10 * SCALING_FACTOR)
GOAL_DETECTION_RADIUS = DOT_RADIUS + TARGET_RADIUS
GAZE_CIRCLE_RADIUS = 30
GAZE_CIRCLE_COLOR = (100, 100, 100, 120)
GHOST_TRAIL_DURATION = 3.0
recent_positions = []
last_reset_time = time.time()
RECENT_DIR_LOOKBACK = 1.0
GOAL_SWITCH_THRESHOLD = 0.05
GAME_CENTER = (GAME_AREA_X + GAME_AREA_SIZE[0] // 2, GAME_AREA_Y + GAME_AREA_SIZE[1] // 2)
START_POS = [GAME_CENTER[0], GAME_CENTER[1]]
dot_pos = START_POS.copy()
gamma = 0.2
reached_goal = False
targets = []
current_target_idx = 0
obstacles = []
goal_counters = {}
failure_counter = 0
wrong_goal_message_time = 0
USE_RAW_ONLY_FOR_GOAL_DETECTION = True
has_started_moving = False
movement_start_time = 0
target_lock_time = 3.0
last_redirect_time = 0
redirect_interval = 5.0
redirect_chance = 0.3
user_intended_target = None
initial_target_selected = False
AXIS_L2 = 4
AXIS_R2 = 5

def create_compatible_surface(size):
    return pygame.Surface(size, flags=pygame.SRCALPHA)

def surface_to_texture(renderer, surf):
    if surf.get_bitsize() != 32:
        surf = surf.convert_alpha()
    return Texture.from_surface(renderer, surf)

def distance(pos1, pos2):
    return math.hypot(pos1[0] - pos2[0], pos1[1] - pos2[1])

def line_circle_intersection(start, end, circle_center, radius):
    dx = end[0] - start[0]
    dy = end[1] - start[1]
    cx = circle_center[0] - start[0]
    cy = circle_center[1] - start[1]
    l2 = dx * dx + dy * dy
    if l2 == 0:
        return distance(start, circle_center) <= radius
    t = max(0, min(1, (cx * dx + cy * dy) / l2))
    proj_x = start[0] + t * dx
    proj_y = start[1] + t * dy
    return distance((proj_x, proj_y), circle_center) <= radius

def check_collision(pos, new_pos):
    if not ENABLE_OBSTACLES: return False
    for obstacle_pos in obstacles:
        if line_circle_intersection(pos, new_pos, obstacle_pos, OBSTACLE_RADIUS + COLLISION_BUFFER):
            return True
    return False

def inside_obstacle(pos):
    if not ENABLE_OBSTACLES: return False
    for obstacle_pos in obstacles:
        if distance(pos, obstacle_pos) <= (OBSTACLE_RADIUS + DOT_RADIUS):
            return True
    return False

def get_recent_direction():
    if len(recent_positions) < 2: return [0, 0]
    current_time = time.time()
    valid_points = [p for p in reversed(recent_positions) if (current_time - p[2]) <= RECENT_DIR_LOOKBACK]
    if len(valid_points) < 2: return [0, 0]
    valid_points.sort(key=lambda p: p[2])
    x1, y1, t1 = valid_points[0]
    x2, y2, t2 = valid_points[-1]
    dt = t2 - t1
    if dt < 0.001: return [0, 0]
    vx, vy = (x2 - x1) / dt, (y2 - y1) / dt
    mag = math.hypot(vx, vy)
    return [vx / mag, vy / mag] if mag > 0 else [0, 0]

def compute_perfect_direction(dot_pos, goal_pos, obstacles):
    gx, gy = goal_pos[0] - dot_pos[0], goal_pos[1] - dot_pos[1]
    goal_dist = math.hypot(gx, gy)
    if goal_dist < 1e-6: return [0, 0]
    goal_dir = [gx / goal_dist, gy / goal_dist]
    repulse_x, repulse_y = 0.0, 0.0
    repulsion_radius, repulsion_gain = 27 * SCALING_FACTOR, 30000.0
    for obs in obstacles:
        dx, dy = dot_pos[0] - obs[0], dot_pos[1] - obs[1]
        dist_o = math.hypot(dx, dy)
        if 1e-6 < dist_o < repulsion_radius:
            push_dir_x, push_dir_y = dx / dist_o, dy / dist_o
            strength = repulsion_gain / (dist_o ** 2)
            repulse_x += push_dir_x * strength
            repulse_y += push_dir_y * strength
    w_px, w_py = goal_dir[0] + repulse_x, goal_dir[1] + repulse_y
    norm = math.hypot(w_px, w_py)
    return [w_px / norm, w_py / norm] if norm > 1e-6 else [0, 0]

def draw_gamma_gauge(surface, gamma_value, x, y, width=150, height=80):
    pygame.draw.rect(surface, LIGHT_GRAY, (x, y, width, height), 0)
    pygame.draw.rect(surface, DARK_GRAY, (x, y, width, height), 2)
    gauge_title = font.render("Assistance", True, TEXT_COLOR)
    title_rect = gauge_title.get_rect(center=(x + width // 2, y + 15))
    surface.blit(gauge_title, title_rect)
    bar_height, bar_y, bar_width, bar_x = 20, y + 35, width - 20, x + 10
    pygame.draw.rect(surface, GRAY, (bar_x, bar_y, bar_width, bar_height))
    if gamma_value > 0:
        fill_width = int(bar_width * gamma_value)
        if gamma_value < 0.5:
            color = (int(255 * (gamma_value * 2)), 180, 60)
        else:
            color = (255, int(180 - (gamma_value - 0.5) * 2 * 120), 60)
        pygame.draw.rect(surface, color, (bar_x, bar_y, fill_width, bar_height))
    pygame.draw.rect(surface, DARK_GRAY, (bar_x, bar_y, bar_width, bar_height), 2)
    if gamma_value > 0:
        marker_x = bar_x + int(bar_width * gamma_value)
        pygame.draw.line(surface, BLACK, (marker_x, bar_y - 3), (marker_x, bar_y + bar_height + 3), 2)
    value_text = font.render(f"{gamma_value:.2f}", True, TEXT_COLOR)
    value_rect = value_text.get_rect(center=(x + width // 2, bar_y + bar_height + 15))
    surface.blit(value_text, value_rect)

def draw_controller_guide(surface, x, y, width=200, height=160):
    pygame.draw.rect(surface, LIGHT_GRAY, (x, y, width, height), 0)
    pygame.draw.rect(surface, DARK_GRAY, (x, y, width, height), 2)
    guide_title = font.render("Controller Guide", True, TEXT_COLOR)
    title_rect = guide_title.get_rect(center=(x + width // 2, y + 15))
    surface.blit(guide_title, title_rect)
    text_x, text_y, spacing = x + 20, y + 40, 35
    control_text = font.render("Left Stick - Move Dot", True, TEXT_COLOR)
    surface.blit(control_text, (text_x, text_y))
    text_y += spacing
    reset_text = font.render("Square - Reset Position", True, TEXT_COLOR)
    surface.blit(reset_text, (text_x, text_y))
    text_y += spacing
    if current_gamma_mode == "manual":
        assist_text = font.render("L2/R2 - Change Assistance", True, TEXT_COLOR)
        surface.blit(assist_text, (text_x, text_y))

class GammaPredictor:
    def __init__(self, model_path="gamma_ppo_model.zip"):
        try:
            self.model = PPO.load(model_path)
        except:
            self.model = None
        self.max_dist = np.sqrt(GAME_AREA_SIZE[0] ** 2 + GAME_AREA_SIZE[1] ** 2)
    def prepare_observation(self, dot_pos, target_pos, human_input):
        dot_pos = np.array(dot_pos, dtype=np.float32)
        target_pos = np.array(target_pos, dtype=np.float32)
        to_target = target_pos - dot_pos
        dist = np.linalg.norm(to_target)
        perfect_dir = to_target / dist if dist > 0 else np.array([0, 0], dtype=np.float32)
        h_mag = np.linalg.norm(human_input)
        human_dir = human_input / h_mag if h_mag > 0 else np.array([0, 0], dtype=np.float32)
        normalized_dist = dist / self.max_dist
        obs_dist_ratio = 1.0
        obs = np.concatenate([dot_pos, human_dir, target_pos, perfect_dir, [normalized_dist], [obs_dist_ratio]]).astype(np.float32)
        return obs
    def predict_gamma(self, dot_pos, target_pos, human_input):
        if self.model is None: return 0.2
        obs = self.prepare_observation(dot_pos, target_pos, human_input)
        obs_batched = obs[np.newaxis, :]
        action, _ = self.model.predict(obs_batched, deterministic=True)
        return float(action[0])

gamma_predictor = GammaPredictor()

def predict_human_target(human_input):
    global current_target_idx
    if not targets: return 0
    dist_to_current = distance(dot_pos, targets[current_target_idx])
    if dist_to_current < GOAL_DETECTION_RADIUS * 2: return current_target_idx
    if human_input[0] == 0 and human_input[1] == 0: return current_target_idx
    h_mag = math.hypot(human_input[0], human_input[1])
    if h_mag <= 1e-6: return current_target_idx
    h_dir = [h / h_mag for h in human_input]
    recent_dir = [0, 0] if USE_RAW_ONLY_FOR_GOAL_DETECTION else get_recent_direction()
    best_score, best_idx = float('-inf'), current_target_idx
    for i, targ in enumerate(targets):
        to_tx, to_ty = targ[0] - dot_pos[0], targ[1] - dot_pos[1]
        to_mag = math.hypot(to_tx, to_ty)
        if to_mag == 0: continue
        to_dir = [to_tx / to_mag, to_ty / to_mag]
        align_human = h_dir[0] * to_dir[0] + h_dir[1] * to_dir[1]
        align_recent = recent_dir[0] * to_dir[0] + recent_dir[1] * to_dir[1]
        score = (align_human * 0.8) + (align_recent * 0.2)
        if score > best_score:
            best_score, best_idx = score, i
    return best_idx

def find_alternative_goal(intended_idx, human_dir):
    if len(targets) <= 1 or intended_idx is None: return 0
    intended_target = targets[intended_idx]
    intended_dir = [intended_target[0] - dot_pos[0], intended_target[1] - dot_pos[1]]
    mag = math.hypot(intended_dir[0], intended_dir[1])
    if mag > 0: intended_dir = [d / mag for d in intended_dir]
    candidates = []
    for i, target in enumerate(targets):
        if i == intended_idx: continue
        target_dir = [target[0] - dot_pos[0], target[1] - dot_pos[1]]
        tmag = math.hypot(target_dir[0], target_dir[1])
        if tmag > 0: target_dir = [d / tmag for d in target_dir]
        dot_product = intended_dir[0] * target_dir[0] + intended_dir[1] * target_dir[1]
        if dot_product > 0.5: candidates.append((i, dot_product))
    if candidates:
        candidates.sort(key=lambda x: x[1], reverse=True)
        return candidates[0][0]
    options = [i for i in range(len(targets)) if i != intended_idx]
    return random.choice(options) if options else 0

data_log = []
trial_start_time = time.time()
current_trajectory = []
trial_outcome = None
save_folder = "user_study_data"
os.makedirs(save_folder, exist_ok=True)

def get_save_filename(seed):
    session_timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    return os.path.join(save_folder, f"data_log_seed{seed}_{session_timestamp}.json")

save_filename = None

def save_data_log(seed):
    if not save_filename: return
    with open(save_filename, "w") as f:
        json.dump(data_log, f, indent=4)
    print(f"Data log updated and saved to {save_filename} for seed={seed}")

def get_mode_number(gamma_mode):
    if gamma_mode_index is not None: return gamma_mode_index + 1
    if isinstance(gamma_mode, float):
        if gamma_mode < 0.1: return 1
        elif gamma_mode < 0.6: return 2
        else: return 3
    elif gamma_mode == "manual": return 4
    elif gamma_mode == "ai": return 5
    return 0

def move_dot(human_input):
    global dot_pos, gamma, reached_goal, current_target_idx, USE_AI_CONTROL, trial_outcome, has_started_moving, movement_start_time, last_redirect_time, user_intended_target, initial_target_selected, expected_goal_idx, wrong_goal_message_time, failure_counter, current_gamma_mode
    if 'wrong_goal_time' not in globals():
        global wrong_goal_time, wrong_goal_correcting, wrong_goal_correction_end
        wrong_goal_time, wrong_goal_correcting, wrong_goal_correction_end = 0, False, 0
    if len(targets) == 0: return [0, 0], [0, 0], [0, 0]
    h_dx, h_dy = human_input
    h_mag = math.hypot(h_dx, h_dy)
    h_dir = [h_dx / h_mag, h_dy / h_mag] if h_mag > 0 else [0, 0]
    if h_mag <= 1e-6: return [0, 0], [0, 0], [0, 0]
    if not has_started_moving and h_mag > 0:
        has_started_moving, movement_start_time = True, time.time()
        if not initial_target_selected:
            proposed_idx = predict_human_target(human_input)
            current_target_idx = proposed_idx if proposed_idx < len(targets) else 0
            user_intended_target = current_target_idx
            initial_target_selected = True
            print(f"Initial user target set to: {user_intended_target}")
    if current_target_idx >= len(targets): current_target_idx = 0
    if expected_goal_idx >= len(targets): expected_goal_idx = 0
    now = time.time()
    if current_target_idx != expected_goal_idx and not wrong_goal_correcting:
        wrong_goal_time += 1 / 60
    else: wrong_goal_time = 0
    if wrong_goal_correcting and now > wrong_goal_correction_end:
        wrong_goal_correcting = False
        print("IDA: Correction period ended, returning control to user")
    if wrong_goal_correcting and current_gamma_mode == 0.5:
        target_pos = targets[expected_goal_idx]
    else: target_pos = targets[current_target_idx]
    w_dir = compute_perfect_direction(dot_pos, target_pos, obstacles)
    input_mag = min(max(h_mag / MAX_SPEED, 0), 1)
    step_size = MAX_SPEED * input_mag

    if current_gamma_mode == 0.5:
        ida_gamma = 0.0
        collision_imminent = False
        for obs in obstacles:
            h_move_x, h_move_y = h_dir[0] * step_size, h_dir[1] * step_size
            potential_pos = [dot_pos[0] + h_move_x, dot_pos[1] + h_move_y]
            if distance(potential_pos, obs) < (OBSTACLE_RADIUS + DOT_RADIUS) * 1.5:
                collision_imminent = True
                break
        universally_worse = True
        for i, goal in enumerate(targets):
            to_goal = [goal[0] - dot_pos[0], goal[1] - dot_pos[1]]
            goal_dist = math.hypot(to_goal[0], to_goal[1])
            if goal_dist > 0:
                goal_dir = [d / goal_dist for d in to_goal]
                if h_dir[0] * goal_dir[0] + h_dir[1] * goal_dir[1] > -0.3:
                    universally_worse = False
                    break
        if wrong_goal_time > 3.0 and not wrong_goal_correcting:
            wrong_goal_correcting, wrong_goal_correction_end = True, now + 0.225
            print(f"IDA: User going to wrong goal for {wrong_goal_time:.1f}s, providing brief nudge")
        if collision_imminent or universally_worse or wrong_goal_correcting:
            if wrong_goal_correcting:
                ida_gamma = 0.8
                target_pos = targets[expected_goal_idx]
                w_dir = compute_perfect_direction(dot_pos, target_pos, obstacles)
                print("IDA mode: Brief correction nudge active")
            else:
                ida_gamma = 1.0
                print("IDA mode: Full intervention activated")
        gamma = ida_gamma
    elif current_gamma_mode == 1.0:
        base_gamma = 0.25
        dist_to_target = distance(dot_pos, target_pos)
        goal_threshold = GOAL_DETECTION_RADIUS * 4
        if dist_to_target < goal_threshold:
            goal_factor = 1.0 - (dist_to_target / goal_threshold)
            base_gamma = max(base_gamma, 0.25 + 0.25 * goal_factor)
        min_obs_distance = min(distance(dot_pos, obs) for obs in obstacles) if obstacles else float('inf')
        obs_threshold = (OBSTACLE_RADIUS + DOT_RADIUS) * 2.5
        if min_obs_distance < obs_threshold:
            obs_factor = 1.0 - (min_obs_distance / obs_threshold)
            base_gamma = max(base_gamma, 0.3 + 0.3 * obs_factor)
        if h_mag > 0.3:
            proposed_idx = predict_human_target(human_input)
            if proposed_idx != current_target_idx and proposed_idx < len(targets):
                proposed_goal = targets[proposed_idx]
                to_proposed = [proposed_goal[0] - dot_pos[0], proposed_goal[1] - dot_pos[1]]
                p_dist = math.hypot(to_proposed[0], to_proposed[1])
                if p_dist > 0:
                    p_dir = [d / p_dist for d in to_proposed]
                    if h_dir[0] * p_dir[0] + h_dir[1] * p_dir[1] > 0.8:
                        current_target_idx = proposed_idx
                        print(f"Mode 3: Detected goal change to {current_target_idx + 1}")
                        base_gamma = max(0.1, base_gamma - 0.2)
        if random.random() < 0.0002 and len(targets) > 1 and dist_to_target > goal_threshold:
            options = [i for i in range(len(targets)) if i != current_target_idx]
            if options:
                current_target_idx = random.choice(options)
                print(f"Mode 3: Randomly switched to goal {current_target_idx + 1}")
        alignment = h_dir[0] * w_dir[0] + h_dir[1] * w_dir[1]
        if alignment > 0.7:
            base_gamma = max(0.1, base_gamma - 0.15)
        gamma = max(0.1, min(0.60, base_gamma + random.uniform(-0.05, 0.05)))
    elif USE_AI_CONTROL and h_mag > 0:
        dist_to_target = distance(dot_pos, target_pos)
        min_obs_distance = min(distance(dot_pos, obs) for obs in obstacles) if obstacles else float('inf')
        goal_threshold = GOAL_DETECTION_RADIUS * 6
        obs_threshold = (OBSTACLE_RADIUS + DOT_RADIUS) * 3
        base_gamma = 0.35
        goals_in_direction = 0
        if h_mag > 0:
            h_dir_norm = [h / h_mag for h in human_input]
            for targ in targets:
                to_goal = [targ[0] - dot_pos[0], targ[1] - dot_pos[1]]
                to_goal_mag = math.hypot(to_goal[0], to_goal[1])
                if to_goal_mag > 0:
                    to_goal_dir = [d / to_goal_mag for d in to_goal]
                    if h_dir_norm[0] * to_goal_dir[0] + h_dir_norm[1] * to_goal_dir[1] > 0.7:
                        goals_in_direction += 1
        if goals_in_direction == 1:
            extended_threshold = goal_threshold * 1.5
            if dist_to_target < extended_threshold:
                goal_factor = 1.0 - (dist_to_target / extended_threshold)
                base_gamma = max(base_gamma, 0.45 + 0.5 * goal_factor)
        else:
            if dist_to_target < goal_threshold:
                goal_factor = 1.0 - (dist_to_target / goal_threshold)
                base_gamma = max(base_gamma, 0.35 + 0.4 * goal_factor)
        if min_obs_distance < obs_threshold:
            obs_factor = 1.0 - (min_obs_distance / obs_threshold)
            base_gamma = max(base_gamma, 0.45 + 0.5 * obs_factor)
        if dist_to_target < goal_threshold and min_obs_distance < obs_threshold:
            base_gamma = max(base_gamma, 0.7 + (0.1 if goals_in_direction == 1 else 0))
        near_end_of_trajectory = dist_to_target < GOAL_DETECTION_RADIUS * 1.5
        if (near_end_of_trajectory or min_obs_distance < (OBSTACLE_RADIUS + DOT_RADIUS) * 1.2) and gamma > 0.7:
            if h_mag > 0.3:
                proposed_idx = predict_human_target(human_input)
                if proposed_idx != current_target_idx and proposed_idx < len(targets):
                    proposed_goal = targets[proposed_idx]
                    to_proposed = [proposed_goal[0] - dot_pos[0], proposed_goal[1] - dot_pos[1]]
                    p_dist = math.hypot(to_proposed[0], to_proposed[1])
                    if p_dist > 0:
                        p_dir = [d / p_dist for d in to_proposed]
                        if h_dir[0] * p_dir[0] + h_dir[1] * p_dir[1] > (0.75 if near_end_of_trajectory else 0.85):
                            current_target_idx = proposed_idx
                            print(f"Mode 5: Goal switch to {current_target_idx + 1} at end of trajectory")
                            base_gamma = 0.3
        gamma = max(0.0, min(1.0, base_gamma + random.uniform(-0.05, 0.05)))
    
    if h_mag > 0.2:
        proposed_idx = predict_human_target(human_input)
        if proposed_idx != current_target_idx and proposed_idx < len(targets):
            proposed_goal = targets[proposed_idx]
            to_proposed = [proposed_goal[0] - dot_pos[0], proposed_goal[1] - dot_pos[1]]
            p_dist = math.hypot(to_proposed[0], to_proposed[1])
            if p_dist > 0:
                p_dir = [d / p_dist for d in to_proposed]
                if h_dir[0] * p_dir[0] + h_dir[1] * p_dir[1] > (0.6 if gamma > HIGH_GAMMA_THRESHOLD else 0.7):
                    current_target_idx = proposed_idx
                    print(f"Goal switched to {current_target_idx + 1}")
                    target_pos = targets[current_target_idx]
                    w_dir = compute_perfect_direction(dot_pos, target_pos, obstacles)

    if gamma > 0.9:
        w_move_x, w_move_y = 0.95 * w_dir[0] * step_size, 0.95 * w_dir[1] * step_size
        h_move_x, h_move_y = 0.05 * h_dir[0] * step_size, 0.05 * h_dir[1] * step_size
    else:
        w_move_x, w_move_y = gamma * w_dir[0] * step_size, gamma * w_dir[1] * step_size
        noise_x, noise_y = np.random.normal(0, NOISE_MAGNITUDE), np.random.normal(0, NOISE_MAGNITUDE)
        noisy_dx, noisy_dy = h_dir[0] + noise_x, h_dir[1] + noise_y
        nm = math.hypot(noisy_dx, noisy_dy)
        if nm > 0: noisy_dx, noisy_dy = noisy_dx / nm, noisy_dy / nm
        h_move_x, h_move_y = (1 - gamma) * noisy_dx * step_size, (1 - gamma) * noisy_dy * step_size

    final_dx = w_move_x + h_move_x
    final_dy = w_move_y + h_move_y
    new_x, new_y = dot_pos[0] + final_dx, dot_pos[1] + final_dy
    if not check_collision(dot_pos, [new_x, new_y]):
        dot_pos[0] = max(GAME_AREA_X, min(GAME_AREA_X + GAME_AREA_SIZE[0], new_x))
        dot_pos[1] = max(GAME_AREA_Y, min(GAME_AREA_Y + GAME_AREA_SIZE[1], new_y))
    if inside_obstacle(dot_pos):
        print("Collision with obstacle -> resetting!")
        trial_outcome, failure_counter = "collision", failure_counter + 1
        reset()
        return [0, 0], [0, 0], [0, 0]
    final_mag = math.hypot(final_dx, final_dy)
    x_dir = [final_dx / final_mag, final_dy / final_mag] if final_mag > 0 else [0, 0]
    dist_to_goal = distance(dot_pos, target_pos)
    if dist_to_goal < GOAL_DETECTION_RADIUS:
        if current_target_idx == expected_goal_idx:
            reached_goal, trial_outcome = True, "success"
            goal_counters[current_target_idx] = goal_counters.get(current_target_idx, 0) + 1
            if goal_counters[current_target_idx] >= 4:
                expected_goal_idx = (expected_goal_idx + 1) % len(targets)
                print(f"Goal {current_target_idx + 1} completed 4 times! Moving to next goal: {expected_goal_idx + 1}")
        else:
            print(f"Wrong goal reached! Expected: {expected_goal_idx + 1}, Reached: {current_target_idx + 1}")
            trial_outcome, failure_counter, wrong_goal_message_time = "wrong_goal", failure_counter + 1, time.time()
            reset()
            return [0, 0], [0, 0], [0, 0]
        pygame.time.set_timer(pygame.USEREVENT, 1000)
    return h_dir, w_dir, x_dir

def reset():
    global dot_pos, reached_goal, current_target_idx, gamma, recent_positions, last_reset_time, trial_start_time, current_trajectory, trial_outcome, failure_counter, has_started_moving, movement_start_time, last_redirect_time, user_intended_target, initial_target_selected
    if trial_start_time is not None and len(current_trajectory) > 0:
        trial_duration = time.time() - trial_start_time
        goal_reached = targets[current_target_idx] if reached_goal and 0 <= current_target_idx < len(targets) else None
        mode = "AI" if USE_AI_CONTROL else "Manual"
        trial_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        if trial_outcome == "manual_reset": failure_counter += 1
        data_log.append({"timestamp": trial_timestamp, "mode": mode, "trial_duration": trial_duration, "trial_outcome": trial_outcome if trial_outcome else "manual_reset", "goal_reached": goal_reached, "trajectory": current_trajectory.copy()})
        print(f"Trial recorded: {data_log[-1]}")
        save_data_log(current_seed)
    dot_pos = START_POS.copy()
    reached_goal = False
    current_target_idx = 0
    gamma = 0.5 if current_gamma_mode == "manual" else 0.95
    recent_positions.clear()
    last_reset_time = time.time()
    trial_start_time = time.time()
    current_trajectory.clear()
    trial_outcome, has_started_moving, movement_start_time, last_redirect_time, user_intended_target, initial_target_selected = None, False, 0, 0, None, False
    pygame.time.set_timer(pygame.USEREVENT, 0)

def initialize_environment_fixed(seed):
    global obstacles, targets, goal_counters, expected_goal_idx
    random.seed(seed)
    np.random.seed(seed)
    obstacles.clear()
    targets.clear()
    margin, min_goal_gap, N_GOALS, N_OBSTACLES = 50 * SCALING_FACTOR, 200 * SCALING_FACTOR, 8, 5
    new_goals, attempts = [], 0
    while len(new_goals) < N_GOALS and attempts < 1000:
        x, y = random.uniform(GAME_AREA_X + margin, GAME_AREA_X + GAME_AREA_SIZE[0] - margin), random.uniform(GAME_AREA_Y + margin, GAME_AREA_Y + GAME_AREA_SIZE[1] - margin)
        candidate = (x, y)
        if all(distance(candidate, g) >= min_goal_gap for g in new_goals):
            new_goals.append(candidate)
        attempts += 1
    targets.extend(new_goals)
    goal_counters, expected_goal_idx = {i: 0 for i in range(len(targets))}, 0
    new_obstacles = []
    obstacle_goals = random.sample(new_goals, k=min(min(N_GOALS - 1, N_OBSTACLES), len(new_goals) - 1)) if len(new_goals) > 1 else new_goals
    for goal in obstacle_goals:
        t = random.uniform(0.6, 0.8)
        base_point = (START_POS[0] + t * (goal[0] - START_POS[0]), START_POS[1] + t * (goal[1] - START_POS[1]))
        vec = (goal[0] - START_POS[0], goal[1] - START_POS[1])
        vec_norm = math.hypot(vec[0], vec[1])
        perp = (-vec[1] / vec_norm, vec[0] / vec_norm) if vec_norm > 1e-6 else (0, 0)
        offset_mag = random.uniform(20 * SCALING_FACTOR, 40 * SCALING_FACTOR)
        offset = (perp[0] * offset_mag * random.choice([-1, 1]), perp[1] * offset_mag * random.choice([-1, 1]))
        candidate = (base_point[0] + offset[0], base_point[1] + offset[1])
        candidate = (max(GAME_AREA_X + margin, min(candidate[0], GAME_AREA_X + GAME_AREA_SIZE[0] - margin)), max(GAME_AREA_Y + margin, min(candidate[1], GAME_AREA_Y + GAME_AREA_SIZE[1] - margin)))
        valid = distance(candidate, START_POS) >= (DOT_RADIUS + OBSTACLE_RADIUS + 10) and distance(candidate, goal) >= (TARGET_RADIUS + OBSTACLE_RADIUS + 20) and all(distance(candidate, obs) >= (2 * OBSTACLE_RADIUS + 10) for obs in new_obstacles)
        if valid: new_obstacles.append(candidate)
    obstacles.extend(new_obstacles)
    print(f"Environment initialized with fixed seed {seed}. #goals={len(targets)}, #obstacles={len(obstacles)}")

def draw_arrow(surface, color, start_pos, direction, length=ARROW_LENGTH):
    dx, dy = direction
    if dx == 0 and dy == 0: return
    mag = math.hypot(dx, dy)
    dx, dy = dx / mag, dy / mag
    end_x, end_y = start_pos[0] + dx * length, start_pos[1] + dy * length
    pygame.draw.line(surface, color, start_pos, (end_x, end_y), int(2 * SCALING_FACTOR))
    arrow_size = 7 * SCALING_FACTOR
    angle = math.atan2(dy, dx)
    arrow1_x, arrow1_y = end_x - arrow_size * math.cos(angle + math.pi / 6), end_y - arrow_size * math.sin(angle + math.pi / 6)
    arrow2_x, arrow2_y = end_x - arrow_size * math.cos(angle - math.pi / 6), end_y - arrow_size * math.sin(angle - math.pi / 6)
    pygame.draw.line(surface, color, (end_x, end_y), (arrow1_x, arrow1_y), int(2 * SCALING_FACTOR))
    pygame.draw.line(surface, color, (end_x, end_y), (arrow2_x, arrow2_y), int(2 * SCALING_FACTOR))

def render_full_view(surface, h_dir, w_dir, x_dir):
    global current_target_idx, wrong_goal_message_time
    surface.fill(BACKGROUND_COLOR)
    game_area_rect = pygame.Rect(GAME_AREA_X, GAME_AREA_Y, GAME_AREA_SIZE[0], GAME_AREA_SIZE[1])
    pygame.draw.rect(surface, LIGHT_GRAY, game_area_rect, 1)
    if ENABLE_OBSTACLES:
        for obstacle_pos in obstacles:
            pygame.draw.circle(surface, GRAY, (int(obstacle_pos[0]), int(obstacle_pos[1])), OBSTACLE_RADIUS)
    for i, target in enumerate(targets):
        goal_color, outline_color, outline_width = (EXPECTED_GOAL_COLOR, BLACK, 2) if i == expected_goal_idx else (YELLOW, None, 0)
        pygame.draw.circle(surface, goal_color, (int(target[0]), int(target[1])), TARGET_RADIUS)
        if outline_color: pygame.draw.circle(surface, outline_color, (int(target[0]), int(target[1])), TARGET_RADIUS + 2, int(outline_width * SCALING_FACTOR))
        num_text = font.render(str(i + 1), True, BLACK)
        surface.blit(num_text, (target[0] - 5, target[1] - 12))
    if len(targets) > 0:
        if current_target_idx >= len(targets): current_target_idx = 0
        curr_t = targets[current_target_idx]
        if current_target_idx != expected_goal_idx:
            segments, radius = 16, TARGET_RADIUS + 5
            for i in range(segments):
                if i % 2 == 0:
                    start_angle, end_angle = i * 2 * math.pi / segments, (i + 1) * 2 * math.pi / segments
                    start_pos = (curr_t[0] + radius * math.cos(start_angle), curr_t[1] + radius * math.sin(start_angle))
                    end_pos = (curr_t[0] + radius * math.cos(end_angle), curr_t[1] + radius * math.sin(end_angle))
                    pygame.draw.line(surface, BLUE, start_pos, end_pos, 2)
        else:
            pygame.draw.circle(surface, BLACK, (int(curr_t[0]), int(curr_t[1])), TARGET_RADIUS + 2, int(2 * SCALING_FACTOR))
    now = time.time()
    while recent_positions and (now - recent_positions[0][2]) > GHOST_TRAIL_DURATION:
        recent_positions.pop(0)
    if len(recent_positions) > 1:
        for idx in range(len(recent_positions) - 1):
            x1, y1, _ = recent_positions[idx]
            x2, y2, _ = recent_positions[idx+1]
            pygame.draw.line(surface, HISTORY_COLOR, (x1, y1), (x2, y2), 2)
    pygame.draw.circle(surface, BLACK, (int(dot_pos[0]), int(dot_pos[1])), DOT_RADIUS, int(2 * SCALING_FACTOR))
    if h_dir != [0, 0]: draw_arrow(surface, BLUE, (int(dot_pos[0]), int(dot_pos[1])), h_dir)
    if w_dir != [0, 0]: draw_arrow(surface, GREEN, (int(dot_pos[0]), int(dot_pos[1])), w_dir)
    if x_dir != [0, 0]: draw_arrow(surface, RED, (int(dot_pos[0]), int(dot_pos[1])), x_dir)
    if USE_GAZE_INPUT and gaze_position:
        circle_surface = pygame.Surface((GAZE_CIRCLE_RADIUS * 2, GAZE_CIRCLE_RADIUS * 2), pygame.SRCALPHA)
        pygame.draw.circle(circle_surface, GAZE_CIRCLE_COLOR, (GAZE_CIRCLE_RADIUS, GAZE_CIRCLE_RADIUS), GAZE_CIRCLE_RADIUS)
        gaze_x, gaze_y = gaze_position
        surface.blit(circle_surface, (gaze_x - GAZE_CIRCLE_RADIUS, gaze_y - GAZE_CIRCLE_RADIUS))
    left_x = 10
    mode_number = get_mode_number(current_gamma_mode)
    mode_title = title_font.render(f"Condition {mode_number}", True, HIGHLIGHT_COLOR)
    surface.blit(mode_title, (left_x, 20))
    y_pos = 60
    g_txt = font.render(f"Gamma: {gamma:.2f}", True, TEXT_COLOR)
    surface.blit(g_txt, (left_x, y_pos))
    y_pos += 30
    form_txt = font.render(f"Movement = {gamma:.2f}W + {1 - gamma:.2f}H", True, TEXT_COLOR)
    surface.blit(form_txt, (left_x, y_pos))
    y_pos += 40
    expected_txt = font.render(f"Current Goal: {expected_goal_idx + 1}", True, EXPECTED_GOAL_COLOR)
    surface.blit(expected_txt, (left_x, y_pos))
    y_pos += 30
    if expected_goal_idx < len(targets) and expected_goal_idx in goal_counters:
        progress_txt = font.render(f"Progress: {goal_counters[expected_goal_idx]}/4", True, TEXT_COLOR)
        surface.blit(progress_txt, (left_x, y_pos))
        y_pos += 30
    controls = ["[ / ]: noise", "R: reset", f"Noise σ: {NOISE_MAGNITUDE:.2f}", f"Control: {'AI' if USE_AI_CONTROL else 'Manual'}", f"Input: {'Gaze' if USE_GAZE_INPUT else ('Force Sensor' if USE_FORCE_SENSOR else 'Joystick/Keyboard')}"]
    for control in controls:
        ctrl_txt = font.render(control, True, TEXT_COLOR)
        surface.blit(ctrl_txt, (left_x, y_pos))
        y_pos += 30
    if not FORCE_SENSOR_AVAILABLE and USE_FORCE_SENSOR:
        unavail_txt = font.render("Force sensor not available!", True, RED)
        surface.blit(unavail_txt, (left_x, y_pos))
        y_pos += 30
    y_pos += 20
    elapsed_time = time.time() - last_reset_time
    timer_text = font.render(f"Time: {elapsed_time:.1f}s", True, TEXT_COLOR)
    surface.blit(timer_text, (left_x, y_pos))
    y_pos += 30
    seed_txt = font.render(f"Scenario Seed: {current_seed}", True, TEXT_COLOR)
    surface.blit(seed_txt, (left_x, y_pos))
    y_pos += 30
    legend_y = FULL_VIEW_SIZE[1] - 140
    legend_title = font.render("Legend:", True, TEXT_COLOR)
    surface.blit(legend_title, (left_x, legend_y))
    legend_items = [("Green Arrow: Perfect Path (W)", GREEN), ("Blue Arrow: Human Movement (H)", BLUE), ("Red Arrow: Dot's Movement", RED), ("Gray line: Movement History", LIGHT_GRAY)]
    for i, (label, color) in enumerate(legend_items):
        text = font.render(label, True, color)
        surface.blit(text, (left_x, legend_y + 30 + i * 25))
    right_x = GAME_AREA_X + GAME_AREA_SIZE[0] + 10
    results_title = font.render("Results:", True, TEXT_COLOR)
    surface.blit(results_title, (right_x, 20))
    result_y = 60
    for i in range(len(targets)):
        count = goal_counters.get(i, 0)
        result_txt = font.render(f"Goal {i + 1}: {count}/4", True, EXPECTED_GOAL_COLOR if i == expected_goal_idx else GREEN)
        surface.blit(result_txt, (right_x, result_y))
        result_y += 30
    failures_txt = font.render(f"Failures: {failure_counter}", True, RED)
    surface.blit(failures_txt, (right_x, result_y))
    gauge_x, gauge_y = right_x, result_y + 40
    draw_gamma_gauge(surface, gamma, gauge_x, gauge_y, 150, 80)
    if reached_goal:
        r_txt = title_font.render(f"Goal Reached in {elapsed_time:.1f}s!", True, GREEN)
        r_rect = r_txt.get_rect(center=(GAME_CENTER[0], 80))
        surface.blit(r_txt, r_rect)
    now = time.time()
    if now - wrong_goal_message_time < 2.0:
        wrong_txt = title_font.render(f"Wrong Goal! Go to Goal {expected_goal_idx + 1}", True, WRONG_GOAL_COLOR)
        wrong_rect = wrong_txt.get_rect(center=(GAME_CENTER[0], 80))
        surface.blit(wrong_txt, wrong_rect)

def render_red_only(surface, x_dir):
    global current_target_idx, wrong_goal_message_time
    surface.fill(BACKGROUND_COLOR)
    game_area_rect = pygame.Rect(GAME_AREA_X, GAME_AREA_Y, GAME_AREA_SIZE[0], GAME_AREA_SIZE[1])
    pygame.draw.rect(surface, LIGHT_GRAY, game_area_rect, 1)
    if ENABLE_OBSTACLES:
        for obstacle_pos in obstacles:
            pygame.draw.circle(surface, GRAY, (int(obstacle_pos[0]), int(obstacle_pos[1])), OBSTACLE_RADIUS)
    now = time.time()
    while recent_positions and (now - recent_positions[0][2]) > GHOST_TRAIL_DURATION:
        recent_positions.pop(0)
    if len(recent_positions) > 1:
        for idx in range(len(recent_positions) - 1):
            x1, y1, _ = recent_positions[idx]
            x2, y2, _ = recent_positions[idx+1]
            pygame.draw.line(surface, HISTORY_COLOR, (x1, y1), (x2, y2), 2)
    for i, target in enumerate(targets):
        color = EXPECTED_GOAL_COLOR if i == expected_goal_idx else RED
        pygame.draw.circle(surface, color, (int(target[0]), int(target[1])), TARGET_RADIUS)
        if i == current_target_idx:
            highlight_color = BLACK if i == expected_goal_idx else BLUE
            pygame.draw.circle(surface, highlight_color, (int(target[0]), int(target[1])), TARGET_RADIUS + 2, int(2 * SCALING_FACTOR))
        num_text = font.render(str(i + 1), True, BLACK)
        surface.blit(num_text, (target[0] - 5, target[1] - 12))
    pygame.draw.circle(surface, BLACK, (int(dot_pos[0]), int(dot_pos[1])), DOT_RADIUS, int(2 * SCALING_FACTOR))
    if x_dir != [0, 0]: draw_arrow(surface, RED, (int(dot_pos[0]), int(dot_pos[1])), x_dir)
    if USE_GAZE_INPUT and gaze_position:
        circle_surface = pygame.Surface((GAZE_CIRCLE_RADIUS * 2, GAZE_CIRCLE_RADIUS * 2), pygame.SRCALPHA)
        pygame.draw.circle(circle_surface, GAZE_CIRCLE_COLOR, (GAZE_CIRCLE_RADIUS, GAZE_CIRCLE_RADIUS), GAZE_CIRCLE_RADIUS)
        gaze_x, gaze_y = gaze_position
        surface.blit(circle_surface, (gaze_x - GAZE_CIRCLE_RADIUS, gaze_y - GAZE_CIRCLE_RADIUS))
    left_x = 10
    mode_number = get_mode_number(current_gamma_mode)
    mode_title = title_font.render(f"Condition {mode_number}", True, HIGHLIGHT_COLOR)
    surface.blit(mode_title, (left_x, 20))
    elapsed_time = time.time() - last_reset_time
    timer_text = font.render(f"Time: {elapsed_time:.1f}s", True, TEXT_COLOR)
    surface.blit(timer_text, (left_x, 60))
    goal_txt = font.render(f"Goal {expected_goal_idx + 1}: {goal_counters.get(expected_goal_idx, 0)}/4", True, EXPECTED_GOAL_COLOR)
    surface.blit(goal_txt, (left_x, 100))
    if USE_FORCE_SENSOR:
        force_text = font.render("Force Sensor Mode", True, TEXT_COLOR)
        surface.blit(force_text, (left_x, 140))
    right_x = GAME_AREA_X + GAME_AREA_SIZE[0] + 10
    results_title = font.render("Results:", True, TEXT_COLOR)
    surface.blit(results_title, (right_x, 20))
    result_y = 60
    for i in range(len(targets)):
        count = goal_counters.get(i, 0)
        result_txt = font.render(f"Goal {i + 1}: {count}/4", True, EXPECTED_GOAL_COLOR if i == expected_goal_idx else GREEN)
        surface.blit(result_txt, (right_x, result_y))
        result_y += 30
    failures_txt = font.render(f"Failures: {failure_counter}", True, RED)
    surface.blit(failures_txt, (right_x, result_y))
    gauge_x, gauge_y = 10, FULL_VIEW_SIZE[1] - 100
    draw_gamma_gauge(surface, gamma, gauge_x, gauge_y, 150, 80)
    if reached_goal:
        completion_text = title_font.render("Goal Reached!", True, GREEN)
        text_rect = completion_text.get_rect(center=(GAME_CENTER[0], 80))
        surface.blit(completion_text, text_rect)
    now = time.time()
    if now - wrong_goal_message_time < 2.0:
        wrong_txt = title_font.render(f"Wrong Goal! Go to Goal {expected_goal_idx + 1}", True, WRONG_GOAL_COLOR)
        wrong_rect = wrong_txt.get_rect(center=(GAME_CENTER[0], 80))
        surface.blit(wrong_txt, wrong_rect)

def skip_to_next_environment():
    global data_log, running, save_filename, current_seed, trial_start_time, failure_counter, gamma_mode_index, gamma_modes, current_gamma_mode, skip_remaining_seeds
    save_data_log(current_seed)
    current_index = SCENARIO_SEEDS.index(current_seed)
    if current_index < len(SCENARIO_SEEDS) - 1:
        print("Skipping to the next environment (same gamma mode).")
        running = False
    else:
        print("Already at the last environment seed. Skipping to next gamma mode.")
        running = False
        skip_remaining_seeds = True

###############################################################################
# MAIN EXPERIMENT LOOP
###############################################################################
gamma_modes = [0.0, "ai"]
current_seed = None
save_filename = None
skip_remaining_seeds = False

for gamma_mode_index, gamma_mode in enumerate(gamma_modes):
    current_gamma_mode = gamma_mode
    print(f"\n===== STARTING GAMMA MODE {gamma_mode_index + 1} = {gamma_mode} =====\n")
    skip_remaining_seeds = False

    for s_index, s in enumerate(SCENARIO_SEEDS):
        if skip_remaining_seeds and s_index > 0:
            print(f"Skipping seed {s} to move to next gamma mode")
            continue

        pygame.init()
        pygame.joystick.init()
        joystick = pygame.joystick.Joystick(0) if pygame.joystick.get_count() > 0 else None
        if joystick:
            joystick.init()
            print("Joystick initialized:", joystick.get_name())
        else:
            print("No joystick detected.")

        window1 = Window("2D Environment: Full View", size=FULL_VIEW_SIZE)
        renderer1 = Renderer(window1, vsync=True)
        window2 = Window("2D Environment: Red Arrow Only", size=RED_ONLY_SIZE)
        renderer2 = Renderer(window2, vsync=True)

        surface_full = create_compatible_surface(FULL_VIEW_SIZE)
        surface_red_only = create_compatible_surface(RED_ONLY_SIZE)
        
        pygame.font.init()
        font = pygame.font.Font(None, FONT_SIZE)
        title_font = pygame.font.Font(None, TITLE_FONT_SIZE)

        if USE_GAZE_INPUT:
            gaze_estimator = RobustGazeEstimator()
            if not run_9_point_calibration_mediapipe(gaze_estimator, window1, renderer1, title_font):
                print("Gaze calibration failed. Shutting down this run.")
                pygame.quit()
                continue
            else:
                gaze_thread = threading.Thread(target=gaze_tracker_thread, daemon=True)
                gaze_thread.start()
                print("👁️ Gaze tracker thread started for this seed.")
        
        current_seed = s
        data_log = []
        save_filename = get_save_filename(s)
        initialize_environment_fixed(s)
        trial_start_time = time.time()
        failure_counter = 0
        expected_goal_idx = 0
        reset()

        running = True
        clock = pygame.time.Clock()

        while running:
            trial_duration = time.time() - trial_start_time
            if trial_duration > 15.0 and not reached_goal:
                print(f"Time limit exceeded ({trial_duration:.1f}s) -> resetting!")
                trial_outcome, failure_counter = "timeout", failure_counter + 1
                reset()

            for event in pygame.event.get():
                if event.type == pygame.QUIT: running = False; break
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_f:
                        USE_FORCE_SENSOR = not USE_FORCE_SENSOR if FORCE_SENSOR_AVAILABLE else False
                        print("Force sensor mode:", USE_FORCE_SENSOR)
                    elif event.key == pygame.K_LEFTBRACKET: NOISE_MAGNITUDE = max(MIN_NOISE, NOISE_MAGNITUDE - NOISE_STEP)
                    elif event.key == pygame.K_RIGHTBRACKET: NOISE_MAGNITUDE = min(MAX_NOISE, NOISE_MAGNITUDE + NOISE_STEP)
                    elif event.key == pygame.K_r: trial_outcome = "manual_reset"; reset()
                    elif event.key == pygame.K_n: skip_to_next_environment()
                    elif event.key == pygame.K_SPACE and USE_FORCE_SENSOR: USE_AI_CONTROL = not USE_AI_CONTROL
                if joystick and event.type == pygame.JOYBUTTONDOWN:
                    if event.button == 2: trial_outcome = "manual_reset"; reset()
                    elif event.button == 3: USE_AI_CONTROL = not USE_AI_CONTROL
                    elif event.button == 4: skip_to_next_environment()
                if event.type == pygame.USEREVENT:
                    if not reached_goal: trial_outcome = "timeout"; reset()

            if not running: break
            
            all_goals_completed = all(goal_counters.get(i, 0) >= 4 for i in range(len(targets)))
            if all_goals_completed and len(targets) > 0:
                print("All goals completed. Moving to next environment..."); running = False; continue
            elif len(targets) == 0:
                print("No targets. Skipping..."); running = False; continue

            if not reached_goal:
                if isinstance(gamma_mode, float):
                    gamma, USE_AI_CONTROL = gamma_mode, False
                elif gamma_mode == "ai":
                    USE_AI_CONTROL = True
                else: # manual
                    USE_AI_CONTROL = False

                dx, dy = 0.0, 0.0
                if USE_GAZE_INPUT:
                    with gaze_lock:
                        if gaze_position:
                            gaze_x, gaze_y = gaze_position
                            dx, dy = gaze_x - dot_pos[0], gaze_y - dot_pos[1]
                elif USE_FORCE_SENSOR and FORCE_SENSOR_AVAILABLE:
                    dx, dy = force_sensor_input
                else:
                    keys = pygame.key.get_pressed()
                    if keys[pygame.K_LEFT]: dx -= 1
                    if keys[pygame.K_RIGHT]: dx += 1
                    if keys[pygame.K_UP]: dy -= 1
                    if keys[pygame.K_DOWN]: dy += 1
                    if joystick:
                        axis_0, axis_1 = joystick.get_axis(0), joystick.get_axis(1)
                        if abs(axis_0) > 0.1 or abs(axis_1) > 0.1:
                            dx, dy = axis_0, axis_1
                        else:
                            dx, dy = 0.0, 0.0
                        if gamma_mode == "manual":
                            if joystick.get_axis(AXIS_L2) > 0.1: gamma = max(0.0, gamma - 0.003)
                            if joystick.get_axis(AXIS_R2) > 0.1: gamma = min(1.0, gamma + 0.003)
                if abs(dx) < 0.1 and abs(dy) < 0.1: dx, dy = 0.0, 0.0
                if not USE_FORCE_SENSOR and not USE_GAZE_INPUT: dx *= MAX_SPEED; dy *= MAX_SPEED
                human_input = [dx, dy]
                if gamma <= HIGH_GAMMA_THRESHOLD:
                    proposed_idx = predict_human_target(human_input)
                    current_target_idx = proposed_idx if proposed_idx < len(targets) else 0
                h_dir, w_dir, x_dir = move_dot(human_input)
                if not reached_goal:
                    recent_positions.append((dot_pos[0], dot_pos[1], time.time()))
                    current_trajectory.append((dot_pos[0], dot_pos[1], gamma))
                else: h_dir, w_dir, x_dir = [0, 0], [0, 0], [0, 0]
            
            render_full_view(surface_full, h_dir, w_dir, x_dir)
            render_red_only(surface_red_only, x_dir)
            
            tex1 = surface_to_texture(renderer1, surface_full)
            tex2 = surface_to_texture(renderer2, surface_red_only)
            renderer1.clear()
            # THE FIX IS HERE: Use texture.draw() for the main loop rendering
            tex1.draw(dstrect=(0, 0, FULL_VIEW_SIZE[0], FULL_VIEW_SIZE[1]))
            renderer1.present()
            renderer2.clear()
            # AND HERE AS WELL
            tex2.draw(dstrect=(0, 0, RED_ONLY_SIZE[0], RED_ONLY_SIZE[1]))
            renderer2.present()
            
            clock.tick(60)
        
        save_data_log(s)
        pygame.quit() 
        print(f"Finished environment seed: {s} (Gamma mode={gamma_mode})")

if ser is not None:
    ser.close()

print("All seeds completed. Exiting.")

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
Error: Could not open serial port: could not open port 'COM5': FileNotFoundError(2, 'The system cannot find the file specified.', None, 2)
Force sensor not available. Using keyboard/joystick controls only.

===== STARTING GAMMA MODE 1 = 0.0 =====

No joystick detected.
Added calibration point 1 at (160.0, 80.0)
Added calibration point 2 at (800.0, 80.0)
Added calibration point 3 at (1440.0, 80.0)
Added calibration point 4 at (160.0, 400.0)
Added calibration point 5 at (800.0, 400.0)
Added calibration point 6 at (1440.0, 400.0)
Added calibration point 7 at (160.0, 720.0)
Added calibration point 8 at (800.0, 720.0)
Added calibration point 9 at (1440.0, 720.0)
✅ Calibration model trained successfully.
👁️ Gaze tracker thread started for this seed.
Environment initialized with fixed seed 0. #goals=5, #obstacles=3
